In [283]:
import os
import json
import h5py
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video
import imageio
from statistics import mean
import robomimic.utils.file_utils as FileUtils
from PIL import Image, ImageDraw, ImageFont
import zarr
from diffusion_policy.common.replay_buffer import ReplayBuffer
from filelock import FileLock
from diffusion_policy.codecs.imagecodecs_numcodecs import register_codecs, Jpeg2k
import pdb
from tqdm import tqdm
import xml.etree.ElementTree as ET


In [7]:
og_redcube_data = h5py.File('/proj/vondrick3/sruthi/robots/diffusion_policy/data/robomimic/datasets/lift/ph/robomimic/datasets/lift/ph/image_abs.hdf5', 'r')
print(og_redcube_data['data']['demo_9'].keys())

<KeysViewHDF5 ['actions', 'dones', 'next_obs', 'obs', 'rewards', 'states']>


In [241]:
trial_hammer_basepath = '/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.09.03/21.23.37_train_diffusion_unet_hybrid_15.00.33_check/checkpoints/epoch=0150-test_mean_score=0.940/unguided/alift_4wredcube_11_14_23_34_15/'
obsdict_agentview = np.load(trial_hammer_basepath + 'obsdict_agentview.npy', 'r')
obsdict_eyeinhand = np.load(trial_hammer_basepath + 'obsdict_eyeinhand.npy', 'r')
obsdict_robot0s = np.load(trial_hammer_basepath + 'obsdict_robot0s.npy', 'r')
rewards = np.load(trial_hammer_basepath + 'rewards.npy', 'r')
states = np.load(trial_hammer_basepath + 'startstates.npy', 'r')
actions = np.load(trial_hammer_basepath + 'actions.npy', 'r')

In [107]:
print('hi', trial_hammer_basepath)
obsdict_agentview = (obsdict_agentview.transpose(0,2,1,4,5,3).reshape(13*8,1008,84,84,3)* 255.0).astype(np.uint8)[:-4]
print('obsdict_agentview', obsdict_agentview.shape)

obsdict_eyeinhand = (obsdict_eyeinhand.transpose(0,2,1,4,5,3).reshape(13*8,1008,84,84,3)* 255.0).astype(np.uint8)[:-4]
print('obsdict_eyeinhand', obsdict_eyeinhand.shape)

obsdict_robot0s = obsdict_robot0s.transpose(0,2,1,3).reshape(13*8,1008,9)[:-4]
print('obsdict_robot0s', obsdict_robot0s.shape)

actions = np.load(trial_hammer_basepath + 'actions.npy', 'r')
print(actions.shape)
actions = actions[:,:,:8,:].transpose(0,2,1,3).reshape(13*8,1008,7)[:-4]
print('actions', actions.shape)

rewards = rewards.transpose(1,0)
print('rewards', rewards.shape)

states = np.repeat(np.array(states)[np.newaxis, :, :], obsdict_agentview.shape[0], axis=0)
print('states', states.shape)

hi /proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.09.03/21.23.37_train_diffusion_unet_hybrid_15.00.33_check/checkpoints/epoch=0150-test_mean_score=0.940/unguided/alift_4wredcube_11_14_23_34_15/
obsdict_agentview (100, 1008, 84, 84, 3)
obsdict_eyeinhand (100, 1008, 84, 84, 3)
obsdict_robot0s (100, 1008, 9)
(13, 1008, 8, 7)
actions (100, 1008, 7)
rewards (100, 1008)
states (100, 1008, 32)


expected:

hi
obsdict_agentview (100, 1008, 84, 84, 3)
obsdict_eyeinhand (100, 1008, 84, 84, 3)
obsdict_robot0s (100, 1008, 9)
(13, 1008, 8, 7)
actions (100, 1008, 7)
rewards (100, 1008)
states (100, 1008, 32)

In [108]:
trial_hammer_basepath

'/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.09.03/21.23.37_train_diffusion_unet_hybrid_15.00.33_check/checkpoints/epoch=0150-test_mean_score=0.940/unguided/alift_4wredcube_11_14_23_34_15/'

In [102]:
''' CREATE 1 DATASET '''
new_data = {'data':{}}

count = 0
'''
#for high quality dataset
for datapt in range(0,rewards.shape[1]):
    success = int(np.max(rewards[:,datapt]))
    minsuccess = min(np.argwhere(rewards[:,datapt]==1))[0] if success else -1

    if minsuccess>0 and rewards[:,datapt][minsuccess:minsuccess+10].sum()==10:
        stop_demo = minsuccess+16
        new_data['data'][f'demo_{count}'] = {
            'rewards': rewards[:stop_demo,datapt],
            'success': [success],
            'og_pt': [datapt],
            'states': states[:stop_demo,datapt,:],
            'actions': actions[:stop_demo, datapt],
            'obs': {
                'agentview_image': obsdict_agentview[:stop_demo, datapt],
                'robot0_eye_in_hand_image': obsdict_eyeinhand[:stop_demo, datapt],
                'robot0_eef_pos': obsdict_robot0s[:stop_demo, datapt, :3],
                'robot0_eef_quat': obsdict_robot0s[:stop_demo, datapt, 3:7],
                'robot0_gripper_qpos': obsdict_robot0s[:stop_demo, datapt, 7:],
            }
        }
        count+=1 
    else:
        stop_demo = -1
'''

for datapt in range(0,rewards.shape[1]):
    success = np.max(rewards[:,datapt])
    if success>0:
        stop_demo = min(np.argwhere(rewards[:,datapt]==1))[0] + 16
        new_data['data'][f'demo_{count}'] = {
            'rewards': rewards[:stop_demo,datapt],
            'success': [success],
            'og_pt': [datapt],
            'states': states[:stop_demo,datapt,:],
            'actions': actions[:stop_demo, datapt],
            'obs': {
                'agentview_image': obsdict_agentview[:stop_demo, datapt],
                'robot0_eye_in_hand_image': obsdict_eyeinhand[:stop_demo, datapt],
                'robot0_eef_pos': obsdict_robot0s[:stop_demo, datapt, :3],
                'robot0_eef_quat': obsdict_robot0s[:stop_demo, datapt, 3:7],
                'robot0_gripper_qpos': obsdict_robot0s[:stop_demo, datapt, 7:],
            }
        }
        count+=1 
    else:
        stop_demo = -1   
    
# Open HDF5 file and write in the data_dict structure and info
savepath = trial_hammer_basepath+'data_successful_only.hdf5'
f = h5py.File(savepath, 'w')
datagrp = f.create_group('data')


datagrp.attrs['env_args'] = og_redcube_data['data'].attrs['env_args']
datagrp.attrs['total'] = len(new_data['data'])


for demo in new_data['data']:
    demogrp = datagrp.create_group(demo)
    
    actionsdset = demogrp.create_dataset('actions', data = new_data['data'][demo]['actions'])
    rewardsdset = demogrp.create_dataset('rewards', data = new_data['data'][demo]['rewards'])
    successdset = demogrp.create_dataset('success', data = new_data['data'][demo]['success'])
    statesdset = demogrp.create_dataset('states', data = new_data['data'][demo]['states'])
    statesdset = demogrp.create_dataset('og_pt', data = new_data['data'][demo]['og_pt'])

    obsgrp = demogrp.create_group('obs') 
    for grp_name in new_data['data'][demo]['obs']:
        dset = obsgrp.create_dataset(grp_name, data = new_data['data'][demo]['obs'][grp_name])
print('demo done', demo)
f.close()


demo done demo_612


In [48]:
print(trial_hammer_basepath+'data_all.hdf5')
print(h5py.File(trial_hammer_basepath+'data_alll.hdf5')['data'])
print(h5py.File(trial_hammer_basepath+'data_unsuccessful_only.hdf5')['data'])
print(h5py.File(trial_hammer_basepath+'data_successful_only.hdf5')['data'])

/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.09.03/21.23.37_train_diffusion_unet_hybrid_15.00.33_check/checkpoints/epoch=0150-test_mean_score=0.940/unguided/alift_4wredcube_11_6_11_37_51/data_all.hdf5


FileNotFoundError: [Errno 2] Unable to open file (unable to open file: name = '/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.09.03/21.23.37_train_diffusion_unet_hybrid_15.00.33_check/checkpoints/epoch=0150-test_mean_score=0.940/unguided/alift_4wredcube_11_6_11_37_51/data_alll.hdf5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [49]:
'''TEST THE NEW DATASET'''

data = h5py.File(trial_hammer_basepath + 'data_all.hdf5', 'r')
print(data['data'])
video_path = trial_hammer_basepath+'temp.mp4'
video_writer = imageio.get_writer(video_path, fps=20)
idx = 50
demo = f'demo_{idx}'
print('shape', data['data'][demo]['obs']['agentview_image'].shape)
print('success', data['data'][demo]['success'][:])
print('ogpt', data['data'][demo]['og_pt'][0])
for b in  data['data'][demo]['obs']['agentview_image']:
    img = Image.fromarray((b).astype(np.uint8))
    d = ImageDraw.Draw(img)
    d.text( (2,2), str(idx), fill=255)
    
    #-- back to array
    b = np.asarray(img)

    video_writer.append_data(b)
    idx+=1
video_writer.close()
Video(video_path, embed=True)
data.close()

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (84, 84) to (96, 96) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


<HDF5 group "/data" (1008 members)>
shape (43, 84, 84, 3)
success [1.]
ogpt 50


[swscaler @ 0x686b280] Warning: data is not aligned! This can lead to a speed loss


# Testing other datasets

In [72]:
data=h5py.File('/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/2024-04-26_2/demo_gentex_im128_randcams.hdf5', 'r')
ogdata = h5py.File('/proj/vondrick3/sruthi/robots/diffusion_policy/data/robomimic/datasets/lift/ph/image_abs.hdf5')

In [ ]:
for root, _, files in os.walk('/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage'):
    for file in files:
        if file.endswith(".hdf5"):
            try:
                data=h5py.File(os.path.join(root, file), 'r')
            except:
                print('could not find', root,file)
            temp = json.loads(data['data']['demo_1'].attrs['ep_meta'])
            envname=json.loads(data['data'].attrs['env_args'])['env_name']
            if 'PnP' in envname:
                print(  envname, ' ex: ', temp['lang'])
                for oc in temp['object_cfgs']:
                    if 'graspable' in oc and oc['graspable']:
                        print(oc['info']['cat'],oc['info']['mjcf_path'].split('/')[-3:-1],'g',oc.get('obj_groups',''),'e',oc.get('exclude_obj_groups',''),'w',oc.get('washable',''),'m',oc.get('microwavable',''),'c',oc.get('cookable',''),'f',oc.get('freezable',''),'ms',oc.get('max_size',''),'os',oc.get('object_scale',''))
                        print(temp['lang'].split(' ')[2]==oc['info']['cat'])
            # for i in data['data']:
            #     print(i)
            #     if 'ep_meta' in data['data']['demo_1'].attrs and 'lang' in json.loads(data['data']['demo_1'].attrs['ep_meta']):
            #         print(json.loads(data['data'][i].attrs["ep_meta"])['lang'])

In [ ]:
path='/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPStoveToCounter/2024-05-01/demo_gentex_im128_randcams.hdf5'
data=h5py.File(path,'r')
for demo in range(9,10):
    root = ET.fromstring(data['data'][f'demo_{demo}'].attrs['model_file'])
    asset = root.find("asset")
    textures = asset.findall("texture")
    text_dict = {}
    for x in textures:
        text_dict[x.get('name','')] = '/'.join(x.get('file','').split('/')[-4:])
    print(demo)
    print(text_dict)
    print(text_dict['wall_room_wall'])
    print(text_dict['wall_backing_room_wall'])
    print(text_dict['floor_room_wall'])


In [ ]:
path='/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPStoveToCounter/2024-05-01/demo_gentex_im128_randcams.hdf5'
data=h5py.File(path, 'r')
langset = set()
for ep in data['data']:
    print(ep, json.loads(data['data'][ep].attrs['ep_meta'])['lang'])
    langset.add(json.loads(data['data'][ep].attrs['ep_meta'])['lang'])
print('SPACE')
for lang in langset:
    print(lang)
print(len(langset),'/',len(data['data']))

In [ ]:
path='/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPCabToCounter/2024-04-24/demo_gentex_im128_randcams.hdf5'
data=h5py.File(path, 'r')
graspable_objects=[]
for demo in data['data']:
    print('Episode: ', demo, json.loads(data['data'][demo].attrs["ep_meta"])['lang'])
    ep_meta = json.loads(data['data'][demo].attrs['ep_meta'])['object_cfgs']
    for object_type in ep_meta:
        print('GRASPABLE' if 'graspable' in object_type and object_type['graspable'] else 'No', object_type['info']['cat'])
        if 'graspable' in object_type and object_type['graspable']:
            graspable_objects.append(object_type['info']['cat'])
print(graspable_objects)

In [364]:
path='/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/multi_stage/defrosting_food/MicrowaveThawing/2024-05-11/demo_im128.hdf5'
data=h5py.File(path,'r')
demo_keys = sorted(data['data'], key=lambda x: int(x.split('_')[1]))
ep_lens = []
for demo in demo_keys:
    ep_lens.append(data['data'][demo]['actions'].shape[0])
    max_ep=max(max_ep,data['data'][demo]['actions'].shape[0])
print('len',len(ep_lens))
print('max',max(ep_lens))
print('avg',mean(ep_lens))
temp = np.array([683, 763, 723, 744, 617, 642, 730, 715, 669, 708, 713, 711, 659, 699, 684, 612, 782, 739, 776, 717, 730, 651, 690, 760, 837, 654, 658, 603, 758, 705, 678, 602, 670, 652, 795, 648, 845, 664, 673, 698, 906, 675, 726, 881, 717, 734, 695, 822, 645, 702, 716, 748])
temp==ep_lens[:-1]

len 53
max 906
avg 711.0377358490566


array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True])

In [54]:
basepath = '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPStoveToCounter/2024-05-01/demo_gentex_im128_randcams.hdf5'
data=h5py.File(f'{basepath}/demo_gentex_im128_randcams.hdf5', 'r')

for side in ['agentview_left','agentview_right','eye_in_hand']:
    if not os.path.exists(basepath + f'/demos/{side}'):
        os.makedirs(basepath + f'/demos/{side}')
    vidcount=0
    for demo in data['data']:
        if vidcount < 5:
            video_path = f'{basepath}/demos/{side}/{demo}.mp4'
            video_writer = imageio.get_writer(video_path, fps=20)
            if 'success' in data['data'][demo]:
                print('success',data['data'][demo]['success'][0])
            idx=0
            for b in  data['data'][demo]['obs']['robot0_'+side+'_image']:
                img = Image.fromarray((b).astype(np.uint8))
                d = ImageDraw.Draw(img)
                d.text( (2,2), str(idx), fill=255)
                b = np.asarray(img)
                video_writer.append_data(b)
                idx+=1
            video_writer.close()
            Video(video_path, embed=True)
            vidcount+=1

data.close()

NotADirectoryError: [Errno 20] Unable to open file (unable to open file: name = '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPStoveToCounter/2024-05-01/demo_gentex_im128_randcams.hdf5/demo_gentex_im128_randcams.hdf5', errno = 20, error message = 'Not a directory', flags = 0, o_flags = 0)

# Combine Datasets

In [ ]:
temp = h5py.File('/proj/vondrick3/sruthi/robots/diffusion_policy/data/robomimic/datasets/lift/ph/robomimic/datasets/image_abs.hdf5')

In [ ]:
temp['data'].keys()

In [ ]:
''' CREATE 1 DATASET '''
paths = ['/proj/vondrick3/sruthi/robots/diffusion_policy/data/curateddata/greencube2_seed6000/data_all.hdf5', 
         '/proj/vondrick3/sruthi/robots/diffusion_policy/data/curateddata/redcube2_seed6000/data_all.hdf5',
         '/proj/vondrick3/sruthi/robots/diffusion_policy/data/curateddata/hammer2_seed6000/data_all.hdf5',
         '/proj/vondrick3/sruthi/robots/diffusion_policy/data/curateddata/mugbeige2_seed6000/data_all.hdf5',
         '/proj/vondrick3/sruthi/robots/diffusion_policy/data/curateddata/mugred2_seed6000/data_all.hdf5',
         '/proj/vondrick3/sruthi/robots/diffusion_policy/data/curateddata/needle2_seed6000/data_all.hdf5']

# paths= ['/proj/vondrick3/sruthi/robots/diffusion_policy/data/robomimic/datasets/lift/ph/robomimic/datasets/image_abs.hdf5',
#         '/proj/vondrick3/sruthi/robots/diffusion_policy/data/curateddata/needle2/data_successful_only.hdf5']
# Open HDF5 file and write in the data_dict structure and info
base_path = '/proj/vondrick3/sruthi/robots/diffusion_policy/data/curateddata/combined2_seed6000/'
savepath = base_path+'data_all.hdf5'
f = h5py.File(savepath, 'w')
datagrp = f.create_group('data')
datagrp.attrs['env_args'] = og_redcube_data['data'].attrs['env_args']

count=0
for current_path in paths:
    if 'hammer' in current_path:
        current_object = 'hammer'
    elif 'needle' in current_path:
        current_object = 'needle'
    elif 'greencube' in current_path:
        current_object = 'greencube'
    elif 'mugbeige' in current_path:
        current_object = 'mugbeige'
    elif 'robomimic/datasets/image_abs' in current_path:
        current_object = 'redcube'
    current_dataset = h5py.File(current_path)
    for demo in current_dataset['data']:
        demogrp = datagrp.create_group('demo_'+str(count))
        objectsdset = demogrp.create_dataset('object', data = current_object)
        actionsdset = demogrp.create_dataset('actions', data = current_dataset['data'][demo]['actions'])
        rewardsdset = demogrp.create_dataset('rewards', data = current_dataset['data'][demo]['rewards'])
        if current_object == 'redcube':
            successdset = demogrp.create_dataset('success', data = [1.0])
        else:
            successdset = demogrp.create_dataset('success', data = current_dataset['data'][demo]['success'])
        statesdset = demogrp.create_dataset('states', data = current_dataset['data'][demo]['states'])
        # statesdset = demogrp.create_dataset('og_pt', data = current_dataset['data'][demo]['og_pt'])
        obsgrp = demogrp.create_group('obs') 
        for grp_name in current_dataset['data'][demo]['obs']:
            dset = obsgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['obs'][grp_name])
        print('demo done', count)
        count += 1

    print('dataset done', current_path)

datagrp.attrs['total'] = count
f.close()

In [140]:
'''TEST THE NEW DATASET'''

data = h5py.File(savepath, 'r')
print(data['data'])
video_path = base_path+'temp1.mp4'
video_writer = imageio.get_writer(video_path, fps=20)
idx = 6040
demo = f'demo_{idx}'
print(data['data'][demo]['object'])
print(data['data'][demo]['obs']['agentview_image'].shape)
print(data['data'][demo]['success'][:])
for b in  data['data'][demo]['obs']['agentview_image']:
    img = Image.fromarray((b).astype(np.uint8))
    d = ImageDraw.Draw(img)
    d.text( (2,2), str(idx), fill=255)
    
    
    #-- back to array
    b = np.asarray(img)

    video_writer.append_data(b)
    idx+=1
video_writer.close()
Video(video_path, embed=True)
data.close()

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (84, 84) to (96, 96) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


<HDF5 group "/data" (6048 members)>
<HDF5 dataset "object": shape (), type "|O">
(100, 84, 84, 3)
[0.]


[swscaler @ 0x569b080] Warning: data is not aligned! This can lead to a speed loss
